<a href="https://colab.research.google.com/github/jarifrahaman/BERT_fine_tuned_mode/blob/main/BERT_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets torch scikit-learn accelerate

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import numpy as np
import random

# Reproducibility — fixing seeds means results are consistent across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
dataset = load_dataset("dair-ai/emotion")

label_names = dataset["train"].features["label"].names
print("Classes:", label_names)
print(dataset)

# Peek at a few examples
for i in range(5):
    ex = dataset["train"][i]
    print(f"[{label_names[ex['label']]}] {ex['text']}")


Classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
[sadness] i didnt feel humiliated
[sadness] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
[anger] im grabbing a minute to post i feel greedy wrong
[love] i am ever feeling nostalgic about the fireplace i will know that it is still on the property
[anger] i am feeling grouchy


In [ ]:
# Check class distribution in the training set — important for spotting imbalance
from collections import Counter

train_labels = dataset["train"]["label"]
counts = Counter(train_labels)
for label_id, count in sorted(counts.items()):
    print(f"{label_names[label_id]:10s}: {count:5d}  ({count/len(train_labels)*100:.1f}%)")


sadness   :  4666  (29.2%)
joy       :  5362  (33.5%)
love      :  1304  (8.2%)
anger     :  2159  (13.5%)
fear      :  1937  (12.1%)
surprise  :   572  (3.6%)


In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Sanity check: show how one sentence turns into tokens
sample_text = dataset["train"][0]["text"]
tokens = tokenizer.tokenize(sample_text)
print("Original text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", tokenizer.convert_tokens_to_ids(tokens))


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Original text: i didnt feel humiliated
Tokens: ['i', 'didn', '##t', 'feel', 'humiliated']
Token IDs: [1045, 2134, 2102, 2514, 26608]


In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(tokenized_dataset["train"], batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized_dataset["validation"], batch_size=BATCH_SIZE)
test_loader  = DataLoader(tokenized_dataset["test"], batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


Train batches: 1000 | Val batches: 125 | Test batches: 125


In [ ]:
NUM_LABELS = len(label_names)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 109,486,854


In [ ]:
EPOCHS = 4
LEARNING_RATE = 2e-5

# Compute class weights = inverse frequency, so rare classes get a bigger loss penalty
class_counts = np.array([counts[i] for i in range(NUM_LABELS)])
class_weights = (1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * NUM_LABELS  # normalize
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


Class weights: tensor([0.3301, 0.2873, 1.1812, 0.7134, 0.7952, 2.6928], device='cuda:0')


In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(loader)
    return acc, avg_loss, all_preds, all_labels


best_val_acc = 0.0
BEST_MODEL_PATH = "best_bert_emotion_model.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch} | Step {step}/{len(train_loader)} | Batch loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    val_acc, val_loss, _, _ = evaluate(model, val_loader)

    print(f"\n== Epoch {epoch} done == Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val accuracy: {val_acc*100:.2f}%\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> New best model saved (val accuracy {val_acc*100:.2f}%)")

print(f"\nBest validation accuracy achieved: {best_val_acc*100:.2f}%")


Epoch 1 | Step 0/1000 | Batch loss: 1.8513
Epoch 1 | Step 200/1000 | Batch loss: 1.6565
Epoch 1 | Step 400/1000 | Batch loss: 0.8407
Epoch 1 | Step 600/1000 | Batch loss: 0.1424
Epoch 1 | Step 800/1000 | Batch loss: 0.6380

== Epoch 1 done == Train loss: 0.8313 | Val loss: 0.2665 | Val accuracy: 92.50%

  -> New best model saved (val accuracy 92.50%)
Epoch 2 | Step 0/1000 | Batch loss: 0.0383
Epoch 2 | Step 200/1000 | Batch loss: 0.2645
Epoch 2 | Step 400/1000 | Batch loss: 0.2263
Epoch 2 | Step 600/1000 | Batch loss: 0.2457
Epoch 2 | Step 800/1000 | Batch loss: 0.0644

== Epoch 2 done == Train loss: 0.2013 | Val loss: 0.1778 | Val accuracy: 93.25%

  -> New best model saved (val accuracy 93.25%)
Epoch 3 | Step 0/1000 | Batch loss: 0.1752
Epoch 3 | Step 200/1000 | Batch loss: 0.0864
Epoch 3 | Step 400/1000 | Batch loss: 0.1982
Epoch 3 | Step 600/1000 | Batch loss: 0.0109
Epoch 3 | Step 800/1000 | Batch loss: 0.1302

== Epoch 3 done == Train loss: 0.1356 | Val loss: 0.2255 | Val accurac

In [ ]:
def predict_emotion(text, model=model, tokenizer=tokenizer, device=device):
    model.eval()
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    predicted_idx = int(np.argmax(probs))
    predicted_label = label_names[predicted_idx]
    confidence = float(probs[predicted_idx])

    all_scores = {label_names[i]: float(probs[i]) for i in range(len(label_names))}
    return predicted_label, confidence, all_scores

In [ ]:
my_text = "i am not happy today"

label, conf, scores = predict_emotion(my_text)
print(f"Text: {my_text}")
print(f"Predicted emotion: {label}  (confidence: {conf*100:.1f}%)")
for emotion, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {emotion:10s}: {score*100:5.1f}%")

Text: i am not happy today
Predicted emotion: joy  (confidence: 99.9%)
  joy       :  99.9%
  sadness   :   0.0%
  surprise  :   0.0%
  love      :   0.0%
  anger     :   0.0%
  fear      :   0.0%


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import random

In [ ]:
negation_examples = [
    ("i am not happy today", 0),
    ("i am not feeling happy at all", 0),
    ("this is not making me happy", 0),
    ("i don't feel joyful right now", 0),
    ("nothing about this makes me smile", 0),
    ("i am not okay with this", 3),
    ("i am not fine, this really bothers me", 3),
    ("i am not calm about what happened", 3),
    ("i don't trust him anymore", 3),
    ("i am not scared of anything right now", 5),
    ("i am not afraid to try this", 5),
    ("nothing about this worries me", 5),
    ("i am not sad about leaving", 1),
    ("i don't feel down today", 1),
    ("i am not upset with you", 1),
    ("i am not angry, i understand why it happened", 1),
    ("i wasn't expecting that, what a surprise", 5),
    ("i can't believe this happened, i didn't see it coming", 5),
    ("i am not in love with this idea", 0),
    ("this isn't something i love", 0),
    ("i don't feel scared anymore, i feel confident", 1),
    ("i am not worried about the results", 1),
    ("this doesn't upset me at all", 1),
    ("i am not excited about this trip", 0),
    ("i don't feel proud of what i did", 0),
]

print(f"Negation examples: {len(negation_examples)}")

Negation examples: 25


In [ ]:
class NegationDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text, label = self.examples[idx]
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }

negation_dataset = NegationDataset(negation_examples, tokenizer, MAX_LEN)


In [ ]:
original_train = tokenized_dataset["train"]
sample_size = min(1500, len(original_train))
sample_indices = random.sample(range(len(original_train)), sample_size)
original_sample = original_train.select(sample_indices)
original_sample.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Wrap original_sample so it matches the dict format NegationDataset uses
class WrappedHFDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        row = self.ds[idx]
        return {
            "input_ids": row["input_ids"],
            "attention_mask": row["attention_mask"],
            "label": row["label"],
        }

combined_dataset = ConcatDataset([negation_dataset, WrappedHFDataset(original_sample)])
combined_loader = DataLoader(combined_dataset, batch_size=8, shuffle=True)

print(f"Combined fine-tuning set: {len(combined_dataset)} examples "
      f"({len(negation_dataset)} negation + {len(original_sample)} original)")


Combined fine-tuning set: 1525 examples (25 negation + 1500 original)


In [ ]:
FOLLOWUP_LR = 1e-5      # lower than the original 2e-5 — gentle nudge only
FOLLOWUP_EPOCHS = 2

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=FOLLOWUP_LR, weight_decay=0.01)

model.train()
for epoch in range(1, FOLLOWUP_EPOCHS + 1):
    total_loss = 0
    for batch in combined_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(combined_loader)
    print(f"Follow-up epoch {epoch}/{FOLLOWUP_EPOCHS} | avg loss: {avg_loss:.4f}")

print("\nFollow-up fine-tuning done.")


Follow-up epoch 1/2 | avg loss: 0.1587
Follow-up epoch 2/2 | avg loss: 0.0819

Follow-up fine-tuning done.


In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score
new_test_acc = accuracy_score(all_labels, all_preds)
print(f"Test accuracy after follow-up fine-tuning: {new_test_acc*100:.2f}%")
print("Compare this to your original Step 8 test accuracy — it should be similar or better, not lower.")


Test accuracy after follow-up fine-tuning: 92.45%
Compare this to your original Step 8 test accuracy — it should be similar or better, not lower.


In [ ]:
def predict_emotion(text, model=model, tokenizer=tokenizer, device=device):
    model.eval()
    inputs = tokenizer(
        text, padding="max_length", truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    idx = int(probs.argmax())
    return label_names[idx], float(probs[idx]), {label_names[i]: float(probs[i]) for i in range(len(label_names))}

test_sentences = [
    "You scared me",
]

print("\n--- Negation test results ---")
for text in test_sentences:
    label, conf, scores = predict_emotion(text)
    print(f"\nText: {text}")
    print(f"Predicted: {label} ({conf*100:.1f}%)")
    for e, s in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {e:10s}: {s*100:5.1f}%")


--- Negation test results ---

Text: You scared me
Predicted: fear (99.8%)
  fear      :  99.8%
  surprise  :   0.1%
  anger     :   0.0%
  joy       :   0.0%
  love      :   0.0%
  sadness   :   0.0%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/bert-emotion-model"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model and tokenizer saved to {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to /content/drive/MyDrive/bert-emotion-model


In [ ]:
import os
print(os.listdir(SAVE_DIR))

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']
